# 03b Berlin Optimization


In [ ]:
# Install package from GitHub
!pip install git+https://github.com/SilasPignotti/urban-tree-transfer.git -q
# Optional dependencies
!pip install optuna pytorch-tabnet -q

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive

drive.mount('/content/drive')

BASE_DIR = Path('/content/drive/MyDrive/dev/urban-tree-transfer')
DATA_DIR = BASE_DIR / 'data'
OUTPUT_DIR = BASE_DIR / 'outputs/phase_3'

(OUTPUT_DIR / 'metadata').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'figures').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'logs').mkdir(parents=True, exist_ok=True)

from urban_tree_transfer.config import load_experiment_config
from urban_tree_transfer.experiments import (
    ablation,
    data_loading,
    preprocessing,
    training,
    evaluation,
    visualization,
    models,
    transfer,
    hp_tuning,
)
from urban_tree_transfer.utils import (
    validate_setup_decisions,
    validate_algorithm_comparison,
    validate_hp_tuning_result,
    validate_evaluation_metrics,
    validate_finetuning_curve,
)


In [ ]:
algo_path = OUTPUT_DIR / 'metadata/algorithm_comparison.json'
algorithm_comparison = validate_algorithm_comparison(algo_path)

train_df, val_df, test_df = data_loading.load_berlin_splits(DATA_DIR / 'phase_3_experiments')
feature_cols = data_loading.get_feature_columns(train_df)

x_train_scaled, x_val_scaled, x_test_scaled, scaler = preprocessing.scale_features(
    train_df[feature_cols], x_val=val_df[feature_cols], x_test=test_df[feature_cols]
)

y_train, label_to_idx, idx_to_label = preprocessing.encode_genus_labels(train_df['genus_latin'])
y_val = val_df['genus_latin'].map(label_to_idx).to_numpy()
y_test = test_df['genus_latin'].map(label_to_idx).to_numpy()

# Train ML champion
ml_champion = algorithm_comparison['ml_champion']['name']
ml_model = models.create_model(ml_champion)
ml_model = training.train_final_model(ml_model, x_train_scaled, y_train, x_val_scaled, y_val)

# Save model and scaler
model_dir = OUTPUT_DIR / 'models'
model_dir.mkdir(parents=True, exist_ok=True)
model_path = model_dir / 'berlin_ml_champion.pkl'
training.save_model(
    ml_model,
    model_path,
    metadata={
        'feature_columns': feature_cols,
        'label_to_idx': label_to_idx,
        'idx_to_label': idx_to_label,
        'model_name': ml_champion,
    },
)

import pickle
with (model_dir / 'berlin_scaler.pkl').open('wb') as handle:
    pickle.dump(scaler, handle)

# Evaluate
preds = ml_model.predict(x_test_scaled)
metrics = evaluation.compute_metrics(y_test, preds)
per_class = evaluation.compute_per_class_metrics(y_test, preds, list(idx_to_label.values()))
cm = evaluation.compute_confusion_matrix(y_test, preds, labels=list(range(len(idx_to_label))))

metrics_path = OUTPUT_DIR / 'metadata/berlin_evaluation.json'
metrics_path.write_text(json.dumps({'metrics': metrics, 'per_class': per_class.to_dict(orient='records'), 'confusion_matrix': cm.tolist()}, indent=2))
validate_evaluation_metrics(metrics_path)

fig_dir = OUTPUT_DIR / 'figures/berlin_optimization'
fig_dir.mkdir(parents=True, exist_ok=True)
visualization.plot_confusion_matrix(cm, list(idx_to_label.values()), fig_dir / 'berlin_confusion_matrix.png')
visualization.plot_per_genus_f1(
    {row['genus']: row['f1_score'] for row in per_class.to_dict(orient='records')},
    fig_dir / 'per_genus_f1_berlin.png',
)
